# 🚀 Distributed Semiconductor Image Restoration on Kaggle (T4 x2 Dual-GPU)

This notebook trains **NAFNet-SR** using **Dual NVIDIA Tesla T4 GPUs (T4 x2)** with **PyTorch DataParallel**, **Automatic Mixed Precision (AMP FP16)**, **Model EMA**, and **Calibrated Composite Metrology Loss**.


## Step 1: Verify Dual GPU (Tesla T4 x2)


In [ ]:
!nvidia-smi
import torch
print('CUDA Available:', torch.cuda.is_available())
print('GPU Count:', torch.cuda.device_count())
for i in range(torch.cuda.device_count()):
    print(f'  [+] GPU {i}: {torch.cuda.get_device_name(i)}')


## Step 2: Clone Repository (Branch `Kunal`)


In [ ]:
!git clone -b Kunal https://github.com/kmbeddedd/semicon_2026.git
%cd semicon_2026


## Step 3: Install Dependencies


In [ ]:
!pip install -q -r requirements.txt


## Step 4: Auto-Detect & Setup Dataset from Kaggle Input

This cell automatically scans `/kaggle/input/` for unzipped dataset folders or `.zip` archives and links them to `data/`.

In [ ]:
import os, glob, shutil, random, zipfile

print('=== Scanning /kaggle/input for Datasets ===')
input_items = glob.glob('/kaggle/input/**/*', recursive=True)
print(f'Found {len(input_items)} items in /kaggle/input/')
for item in input_items[:10]:
    print('  -', item)
if len(input_items) > 10:
    print(f'  ... and {len(input_items) - 10} more')

os.makedirs('data', exist_ok=True)

# 1. If zip files exist in /kaggle/input, extract them
for z in glob.glob('/kaggle/input/**/*.zip', recursive=True):
    print(f'[+] Extracting zip archive: {z} -> data/...')
    with zipfile.ZipFile(z, 'r') as zip_ref:
        zip_ref.extractall('data')

# 2. If unzipped folders exist in /kaggle/input, create symlinks/copies into data/
for root, dirs, _ in os.walk('/kaggle/input'):
    for d in dirs:
        if 'noisylr' in d.lower() or 'noisy_lr' in d.lower():
            target_train_lr = 'data/train/NoisyLR'
            if not os.path.exists(target_train_lr):
                os.makedirs('data/train', exist_ok=True)
                src_lr = os.path.join(root, d)
                print(f'[+] Linking NoisyLR: {src_lr} -> {target_train_lr}')
                try:
                    os.symlink(src_lr, target_train_lr)
                except Exception:
                    shutil.copytree(src_lr, target_train_lr)
        elif d == 'GT' or d.lower() == 'gt' or 'groundtruth' in d.lower():
            target_train_gt = 'data/train/GT'
            if not os.path.exists(target_train_gt):
                os.makedirs('data/train', exist_ok=True)
                src_gt = os.path.join(root, d)
                print(f'[+] Linking GT: {src_gt} -> {target_train_gt}')
                try:
                    os.symlink(src_gt, target_train_gt)
                except Exception:
                    shutil.copytree(src_gt, target_train_gt)

# 3. Create 10% validation split in data/val if not present
if os.path.exists('data/train/NoisyLR') and not os.path.exists('data/val'):
    random.seed(42)
    os.makedirs('data/val/NoisyLR', exist_ok=True)
    os.makedirs('data/val/GT', exist_ok=True)
    files = sorted(glob.glob('data/train/NoisyLR/*.npy'))
    if not files:
        files = sorted(glob.glob('data/train/NoisyLR/*.*'))
    val_files = random.sample(files, k=max(1, int(len(files) * 0.1)))
    for f in val_files:
        fname = os.path.basename(f)
        shutil.copy(f, os.path.join('data/val/NoisyLR', fname))
        gt_f = os.path.join('data/train/GT', fname)
        if os.path.exists(gt_f):
            shutil.copy(gt_f, os.path.join('data/val/GT', fname))
    print(f'[+] Created Val Split: {len(val_files)} samples in data/val/')

print('=== Final Data Directory Status ===')
print('Train NoisyLR count:', len(glob.glob('data/train/NoisyLR/*.*')))
print('Train GT count:     ', len(glob.glob('data/train/GT/*.*')))
print('Val NoisyLR count:  ', len(glob.glob('data/val/NoisyLR/*.*')))
print('Val GT count:       ', len(glob.glob('data/val/GT/*.*')))


## Step 5: Train NAFNet-SR on Dual T4 GPUs (Batch Size 64: 32 per GPU)


In [ ]:
!python train.py --epochs 100 --batch_size 64 --lr 8e-4 --warmup_epochs 5 --scale 2 --patch_size 0 --num_workers 4 --save_dir /kaggle/working/weights


## Step 6: Evaluate & Benchmark (Single Pass & 8-Fold TTA)


In [ ]:
# Fast production evaluation (< 14ms)
!python eval.py --input_dir data/val/NoisyLR --target_dir data/val/GT --output_dir /kaggle/working/val_restored --weights /kaggle/working/weights/best_model.pt --scale 2 --batch_size 16 --no_tta --check_clean_damage

# 8-Fold TTA evaluation
!python eval.py --input_dir data/val/NoisyLR --target_dir data/val/GT --output_dir /kaggle/working/val_restored_tta --weights /kaggle/working/weights/best_model.pt --scale 2 --batch_size 16
